In [ ]:
import datetime, os
import numpy as np
import cv2, skimage
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.measure import find_contours

print((tf.config.list_physical_devices('GPU')))

In [ ]:
# Loading the data
# Update this path to point to your local copy of the heart slices dataset
Data_1 = np.load('datasets/heartslices_dataset/Data.npy')

"""Data_2 = np.load('Data2.npy')

Data= np.concatenate((Data_1,Data_2),0)"""


In [ ]:
# Train/test split

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split((Data_1[:,:,:,:3]/127.5)-1, Data_1[:,:,:,3:]/255, test_size = 0.1, random_state = 42)
print('X_train shape', X_train.shape)
print('X_train shape', X_test.shape)
print('y_train', y_train.shape)
print('y_test', y_test.shape)
print(np.max(y_test))
plt.imshow(y_test[0,:,:,0])

In [ ]:
# Create a datasets from the NumPy arrays
training_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Create mini-batches of the datasets
batch_size = 32
training_dataset = training_dataset.batch(batch_size)
test_dataset = test_dataset.batch(batch_size)

In [ ]:
del X_train, y_train
print("Original datasets are deleted")

In [ ]:
def double_conv_block(x, n_filters):
  #Conv2D then ReLU activation
  x = tf.keras.layers.Conv2D(n_filters, 3, padding = "same", activation = tf.keras.layers.LeakyReLU(alpha=0.1), kernel_initializer = "glorot_normal")(x)
  #Conv2D then ReLU activation
  x = tf.keras.layers.Conv2D(n_filters, 3, padding = "same", activation = tf.keras.layers.LeakyReLU(alpha=0.1), kernel_initializer = "glorot_normal")(x)
  return x

def downsample_block(x, n_filters):
  f = double_conv_block(x, n_filters)
  p = tf.keras.layers.MaxPool2D(2)(f)
  p = tf.keras.layers.Dropout(0.3)(p)

  return f, p

def upsample_block(x, conv_features, n_filters):
  #upsample
  x = tf.keras.layers.Conv2DTranspose(n_filters, 3, 2, padding="same")(x)
  #concatenate 
  x = tf.keras.layers.concatenate([x, conv_features])
  #dropout
  x = tf.keras.layers.Dropout(0.3)(x)
  #Conv2D twice with leaky ReLU activation
  x = double_conv_block(x, n_filters)

  return x

def build_unet_model():

    # inputs
    inputs = tf.keras.Input(shape=(224,224,3))

    # contracting path - downsample
    # 1 - downsample
    f1, p1 = downsample_block(inputs, 16)
    # 2 - downsample
    f2, p2 = downsample_block(p1, 32)
    # 3 - downsample
    f3, p3 = downsample_block(p2, 64)
    # 4 - downsample
    f4, p4 = downsample_block(p3, 128)

    # 5 middle convolution
    bottleneck = double_conv_block(p4, 256)

    # expanding path - upsample
    # 6 - upsample
    u6 = upsample_block(bottleneck, f4, 128)
    # 7 - upsample
    u7 = upsample_block(u6, f3, 64)
    # 8 - upsample
    u8 = upsample_block(u7, f2, 32)
    # 9 - upsample
    u9 = upsample_block(u8, f1, 16)

    # outputs
    outputs = tf.keras.layers.Conv2D(4, 1, padding="same", activation="sigmoid")(u9)

    # unet model with Keras Functional API
    unet_model = tf.keras.Model(inputs, outputs, name="U-Net")

    return unet_model

In [ ]:
model = build_unet_model()
model.compile(optimizer=tf.keras.optimizers.Adam(),
                  loss='binary_crossentropy',
                  metrics='categorical_accuracy')

In [ ]:
number_of_epochs = 10

train_length = len(training_dataset) #number train samples
batch_size=32
steps_per_epoch = train_length //  32       #integer resulting division

test_length = len(X_test) #number of test samples
validation_steps = test_length //  32       #batch_size        #must not be higer than the actual test lenght becasue we run out of data

model_history = model.fit(training_dataset,
                          epochs=number_of_epochs,
                          validation_data=test_dataset)

#look for the lables if they truly have binary values: yeah they only have 0-1
#set batch size to the x size: working to an extent but loss still blows up

# Save the model
model.save('model.keras')

In [ ]:
def display_learning_curves(history):
    acc = history.history["categorical_accuracy"]
    val_acc = history.history["val_categorical_accuracy"]

    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(number_of_epochs)

    fig = plt.figure(figsize=(12,6))

    plt.subplot(1,2,1)
    plt.plot(epochs_range, acc, label="train accuracy")
    plt.plot(epochs_range, val_acc, label="validataion accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(loc="lower right")

    plt.subplot(1,2,2)
    plt.plot(epochs_range, loss, label="train loss")
    plt.plot(epochs_range, val_loss, label="validataion loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc="upper right")

    fig.tight_layout()
    plt.savefig('curves.png')
    plt.show()
    
# Display learning curves 
display_learning_curves(model.history)

In [ ]:
def calculate_iou(model, X_test, y_test, num_classes=4):
    """Calculate mean IoU across the test set."""
    iou_metric = tf.keras.metrics.IoU(num_classes=num_classes, target_class_ids=list(range(num_classes)))
    predictions = model.predict(X_test)
    # Threshold predictions to binary
    pred_bin = (predictions > 0.5).astype(np.float32)
    iou_metric.update_state(y_test, pred_bin)
    iou = iou_metric.result().numpy()
    print(f"Mean IoU: {iou:.4f}")
    return iou

calculate_iou(model, X_test, y_test)

In [ ]:
def show_predictions(model, X_test, y_test, idx=0):
    """Show input, true mask, and predicted mask for a test sample."""
    image = X_test[idx][np.newaxis, ...]
    pred_mask = model.predict(image)[0]
    display([X_test[idx], y_test[idx,:,:,0], pred_mask[:,:,0]])

show_predictions(model, X_test, y_test, idx=0)

In [ ]:
reconstructed_model = tf.keras.models.load_model('model.keras')

In [ ]:
predicted_masks = model.predict(X_test)
first_mask = predicted_masks[0]
plt.imshow(predicted_masks[0,:,:,0])

In [ ]:
# Turning high probability predictions into a real binary mask
ret, pred_bin = cv2.threshold(first_mask[..., 3], 0.95, 1 , cv2.THRESH_BINARY)
#display([X_test[0,:,:,:], y_test[0,:,:], first_mask[..., 0]])
display([X_test[0,:,:,:], y_test[0,:,:,3], pred_bin])